In [16]:
import pandas as pd

# Mostrar todas las columnas y no cortar las tablas en varias líneas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
pedidos = pd.read_csv(
    'Base de Pedidos.csv',
    parse_dates=['creacion_pedido', 'creacion_transicion', 'rider_notified_at', 'rider_accepted_at',
                 'rider_near_pickup_at', 'rider_picked_up_at', 'rider_near_dropoff_at', 'rider_dropped_off_at'],
    low_memory=False,
)
riders = pd.read_csv('Base de Riders.csv')
# Turnos: inicio/fin planificado y real. Se usa el turno REAL para inicio/fin/duracion_hs
# (las horas vienen con y sin cero adelante, ej. '9:58:10' -> format='mixed')
COLS_FECHA_TUR = ['inicio_turno_planificado','inicio_turno_real','fin_turno_planificado','fin_turno_real']
turnos = pd.read_csv('Base de Turnos.csv')
for col in COLS_FECHA_TUR:
    turnos[col] = pd.to_datetime(turnos[col], format='mixed', errors='coerce')
turnos['rider_id'] = turnos['rider_id'].astype('Int64')
turnos['inicio'] = turnos['inicio_turno_real']
turnos['fin']    = turnos['fin_turno_real']
turnos['duracion_hs']      = (turnos['fin'] - turnos['inicio']).dt.total_seconds() / 3600
turnos['duracion_plan_hs'] = (turnos['fin_turno_planificado'] - turnos['inicio_turno_planificado']).dt.total_seconds() / 3600

print('pedidos:', pedidos.shape)
print('riders: ', riders.shape)
print('turnos: ', turnos.shape)

In [18]:
pedidos.head()

,id_pedido,id_repartidor,id_comercio,creacion_pedido,order_status,transicion,creacion_transicion,transicion_id_repartidor,pu_distance,do_distance,rider_notified_at,rider_accepted_at,rider_near_pickup_at,rider_picked_up_at,rider_near_dropoff_at,rider_dropped_off_at,pre_accept_undispatch,post_accept_undispatchs,pago_base,pago_rendimiento,pago_incentivos
0,+++JGqZUpOr67FuWl/lqKyOkVtseA+BBYnoQWdheTzw=,103835454,12810732,2024-05-09 22:00:10.962501,completed,queued,2024-05-09 22:07:25.925563,101764746,814,1381,2024-05-09 22:14:10.225192,2024-05-09 22:14:31.138958,2024-05-09 22:17:31.837747,2024-05-09 22:18:25.618538,2024-05-09 22:25:16.010187,2024-05-09 22:26:33.871297,1,0,17.293571,2.252143,1.548307
1,+++JGqZUpOr67FuWl/lqKyOkVtseA+BBYnoQWdheTzw=,103835454,12810732,2024-05-09 22:00:10.962501,completed,queued,2024-05-09 22:10:54.668982,57465630,814,1381,2024-05-09 22:14:10.225192,2024-05-09 22:14:31.138958,2024-05-09 22:17:31.837747,2024-05-09 22:18:25.618538,2024-05-09 22:25:16.010187,2024-05-09 22:26:33.871297,1,0,17.293571,2.252143,1.548307
2,+++JGqZUpOr67FuWl/lqKyOkVtseA+BBYnoQWdheTzw=,103835454,12810732,2024-05-09 22:00:10.962501,completed,queued,2024-05-09 22:13:20.483614,108143538,814,1381,2024-05-09 22:14:10.225192,2024-05-09 22:14:31.138958,2024-05-09 22:17:31.837747,2024-05-09 22:18:25.618538,2024-05-09 22:25:16.010187,2024-05-09 22:26:33.871297,1,0,17.293571,2.252143,1.548307
3,+++JGqZUpOr67FuWl/lqKyOkVtseA+BBYnoQWdheTzw=,103835454,12810732,2024-05-09 22:00:10.962501,completed,accepted,2024-05-09 22:14:31.136306,103835454,814,1381,2024-05-09 22:14:10.225192,2024-05-09 22:14:31.138958,2024-05-09 22:17:31.837747,2024-05-09 22:18:25.618538,2024-05-09 22:25:16.010187,2024-05-09 22:26:33.871297,0,0,17.293571,2.252143,1.548307
4,+++JGqZUpOr67FuWl/lqKyOkVtseA+BBYnoQWdheTzw=,103835454,12810732,2024-05-09 22:00:10.962501,completed,near_pickup,2024-05-09 22:17:31.837747,103835454,814,1381,2024-05-09 22:14:10.225192,2024-05-09 22:14:31.138958,2024-05-09 22:17:31.837747,2024-05-09 22:18:25.618538,2024-05-09 22:25:16.010187,2024-05-09 22:26:33.871297,0,0,17.293571,2.252143,1.548307


In [19]:
riders.head()

,id_repartidor,vehicle_name,tasa_aceptacion,avg_batch,tenure_in_weeks
0,126644976,Motocicleta,1.000,8.000000,1.43
1,101576406,Bicicleta,0.750,4.500000,25.57
2,37387812,Motocicleta,0.586,4.333333,103.00
3,10460352,Motocicleta,0.736,3.550000,265.29
4,118851570,Motocicleta,0.653,4.804878,17.86


In [20]:
turnos.head()

,shift_id,rider_id,fecha_inicio,hora_inicio,fecha_fin,hora_fin,inicio,fin,duracion_hs
0,iq3/Ixy6r6k4/sgA2pVhnMsE4idiLjF45I+pgy2oLx0=,139640436,NaT,NaN,NaT,NaN,NaT,NaT,NaN
1,Ev9vWlK4Ayk5jgbBwKCg8JAcnuvFS0bA45v3dr6dBVc=,139640436,NaT,NaN,NaT,NaN,NaT,NaT,NaN
2,wxY292TMExOXgg1EEV+cn+VanmLlSE4X6Wpw6txm/X8=,139640436,NaT,NaN,NaT,NaN,NaT,NaT,NaN
3,J3u6aOaL8vfdFgki6AcT0qaKrY17H9uZV54Cq7VKeGU=,138422418,NaT,NaN,NaT,NaN,NaT,NaT,NaN
4,EwUSvb1mOqSbRXkLugG1igXhSheFp4ZgeWXAmfrKBG0=,138422418,2024-08-23,08:59:28.089000,2024-08-23,14:06:28.226000,2024-08-23 08:59:28.089,2024-08-23 14:06:28.226,5.116705


In [21]:
turnos.info()

<class 'pandas.DataFrame'>
RangeIndex: 176235 entries, 0 to 176234
Data columns (total 9 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   shift_id      176235 non-null  str           
 1   rider_id      176235 non-null  Int64         
 2   fecha_inicio  108429 non-null  datetime64[us]
 3   hora_inicio   108429 non-null  str           
 4   fecha_fin     108429 non-null  datetime64[us]
 5   hora_fin      108429 non-null  str           
 6   inicio        108429 non-null  datetime64[us]
 7   fin           108429 non-null  datetime64[us]
 8   duracion_hs   108429 non-null  float64       
dtypes: Int64(1), datetime64[us](4), float64(1), str(3)
memory usage: 12.3 MB
